# 05 Combined-Features Modeling

Notebook นี้ train และประเมินโมเดลโดยใช้ feature หลายกลุ่มรวมกัน ได้แก่ player stats, team aggregates, champion composition, lane, keystone, summoner spell และ queue type


## Feature Set Overview

Notebook นี้เพิ่มข้อมูล champion/setup และ player-level columns จากไฟล์ merged เข้ากับ aggregate features เพื่อสร้าง input ที่กว้างกว่า notebook 03


In [ ]:
import sys
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier


## 1. Load Data

อ่านไฟล์ `merged_v1.csv` จาก Kaggle input และตรวจ shape ของ dataframe


In [ ]:
DATA_PATH = Path("/kaggle/input/datasets/kiatisakkk/cpe232-lol-10min-dataset/t2_transformed/merged_v1.csv")
df = pd.read_csv(DATA_PATH)
df.shape


## 2. Basic Cleaning

ลบ identifier ที่ไม่ใช้เป็น feature, เก็บเฉพาะ row ที่ target ไม่ขัดกัน, ลบ `RedWin`, และใช้ `BlueWin` เป็น target


In [ ]:
drop_id_cols = ["MatchFk", "Patch"]
df = df.drop(columns=[c for c in drop_id_cols if c in df.columns], errors="ignore")

df = df[df["BlueWin"] != df["RedWin"]].reset_index(drop=True)
df = df.drop(columns=["RedWin"], errors="ignore")

df["BlueWin"] = df["BlueWin"].astype(int)
df.shape


## 2.1 Target And Queue Distribution

แสดงสัดส่วน target และจำนวนข้อมูลในแต่ละ `QueueType` ก่อนเริ่ม encode และสร้าง feature


In [ ]:
target_balance = (
    df["BlueWin"]
    .value_counts()
    .rename_axis("BlueWin")
    .reset_index(name="count")
)
target_balance["ratio"] = target_balance["count"] / target_balance["count"].sum()

display(target_balance)

if "QueueType" in df.columns:
    queue_summary = (
        df["QueueType"]
        .value_counts()
        .rename_axis("QueueType")
        .reset_index(name="count")
    )
    queue_summary["ratio"] = queue_summary["count"] / queue_summary["count"].sum()
    display(queue_summary)


## 3. Encode Champion / Spell / Keystone Features

แปลง champion picks เป็น team-level multi-hot columns (`BlueChamp_*`, `RedChamp_*`) และ one-hot encode categorical setup columns อื่น ๆ


In [ ]:
lane_cols = [f"Lane_P{p}" for p in range(1, 11)]
for c in lane_cols:
    if c in df.columns:
        df[c] = df[c].replace("NONE", "JUNGLE")

# --- Champion team-level multi-hot encoding ---
blue_champ_cols = [f"ChampionFk_P{p}" for p in range(1, 6) if f"ChampionFk_P{p}" in df.columns]
red_champ_cols = [f"ChampionFk_P{p}" for p in range(6, 11) if f"ChampionFk_P{p}" in df.columns]
champion_cols = blue_champ_cols + red_champ_cols

blue_champs = df[blue_champ_cols].astype("Int64").astype(str)
red_champs = df[red_champ_cols].astype("Int64").astype(str)

champion_ids = (
    pd.concat([blue_champs.stack(), red_champs.stack()], axis=0)
    .replace("<NA>", np.nan)
    .dropna()
    .sort_values()
    .unique()
)

champion_features = {}
for champ_id in champion_ids:
    champion_features[f"BlueChamp_{champ_id}"] = blue_champs.eq(champ_id).any(axis=1).astype(int)
    champion_features[f"RedChamp_{champ_id}"] = red_champs.eq(champ_id).any(axis=1).astype(int)

df = pd.concat([df.drop(columns=champion_cols, errors="ignore"), pd.DataFrame(champion_features, index=df.index)], axis=1)

# --- Other categorical ID columns ---
categorical_id_prefixes = (
    "PrimaryKeyStone_",
    "SummonerSpell1_",
    "SummonerSpell2_",
)

categorical_id_cols = [
    c for c in df.columns
    if c.startswith(categorical_id_prefixes)
]

for c in categorical_id_cols:
    df[c] = df[c].astype("Int64").astype(str)

ohe_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
df = pd.get_dummies(df, columns=ohe_cols, dummy_na=False)

print(f"Champion IDs encoded: {len(champion_ids):,}")
print(f"Champion features created: {len(champion_ids) * 2:,}")
print(f"Dataset shape after encoding: {df.shape}")


## 4. Feature Engineering

สร้าง feature ระดับทีมและ difference features เพิ่มจากข้อมูลรายผู้เล่น เช่น gold, CS, kills, KDA, damage และ objectives


In [ ]:
# --- Team-level aggregates ---
blue_players = [f"P{i}" for i in range(1, 6)]
red_players  = [f"P{i}" for i in range(6, 11)]

for stat in ["TotalGold", "MinionsKilled", "kills", "deaths", "assists", "DmgDealt"]:
    blue_cols = [f"{stat}_{p}" for p in blue_players]
    red_cols  = [f"{stat}_{p}" for p in red_players]

    if all(c in df.columns for c in blue_cols + red_cols):
        df[f"Blue_{stat}_sum"] = df[blue_cols].sum(axis=1)
        df[f"Red_{stat}_sum"]  = df[red_cols].sum(axis=1)
        df[f"Blue_{stat}_avg"] = df[blue_cols].mean(axis=1)
        df[f"Red_{stat}_avg"]  = df[red_cols].mean(axis=1)
        df[f"Diff_{stat}"]     = df[f"Blue_{stat}_sum"] - df[f"Red_{stat}_sum"]

def safe_kda(k, d, a):
    return (k + a) / d.replace(0, 1)

if {"Blue_kills_sum", "Blue_deaths_sum", "Blue_assists_sum", "Red_kills_sum", "Red_deaths_sum", "Red_assists_sum"}.issubset(df.columns):
    df["Blue_KDA"] = safe_kda(df["Blue_kills_sum"], df["Blue_deaths_sum"], df["Blue_assists_sum"])
    df["Red_KDA"]  = safe_kda(df["Red_kills_sum"],  df["Red_deaths_sum"],  df["Red_assists_sum"])
    df["Diff_KDA"] = df["Blue_KDA"] - df["Red_KDA"]

for obj in ["BaronKills", "RiftHeraldKills", "DragonKills", "TowerKills"]:
    blue_col = f"Blue{obj}"
    red_col = f"Red{obj}"
    if blue_col in df.columns and red_col in df.columns:
        df[f"Diff_{obj}"] = df[blue_col] - df[red_col]

if "BlueKills" in df.columns and "RedKills" in df.columns:
    df["Diff_Kills"] = df["BlueKills"] - df["RedKills"]

for side, players in [("Blue", blue_players), ("Red", red_players)]:
    gold_cols = [f"TotalGold_{p}" for p in players]
    if all(c in df.columns for c in gold_cols):
        df[f"{side}_GoldStd"] = df[gold_cols].std(axis=1)

df.shape


## 5. Select All Features

เลือกทุก feature หลัง preprocessing ยกเว้น target และสร้าง summary จำนวน feature แยกตามกลุ่ม


In [ ]:
target = "BlueWin"
all_feature_cols = [c for c in df.columns if c != target]

X_all = df[all_feature_cols].copy()
y_all = df[target].copy()

# Fill missing values defensively after feature engineering.
X_all = X_all.replace([np.inf, -np.inf], np.nan)
X_all = X_all.fillna(X_all.median(numeric_only=True))

def feature_group(col):
    if col.startswith(("BlueChamp_", "RedChamp_")):
        return "champion_composition"
    if col.startswith("Lane_"):
        return "lane_position"
    if col.startswith(("PrimaryKeyStone_", "SummonerSpell1_", "SummonerSpell2_")):
        return "setup_runes_spells"
    if col.startswith("QueueType_"):
        return "queue_type"
    if col.startswith("Diff_"):
        return "aggregate_difference"
    if col in {
        "BlueBaronKills", "BlueRiftHeraldKills", "BlueDragonKills", "BlueTowerKills", "BlueKills",
        "RedBaronKills", "RedRiftHeraldKills", "RedDragonKills", "RedTowerKills", "RedKills",
    }:
        return "objective_counts"
    if "_P" in col:
        return "player_level_stats"
    if col.startswith(("Blue_", "Red_")):
        return "team_aggregate"
    return "other"

feature_group_table = (
    pd.Series([feature_group(c) for c in all_feature_cols])
    .value_counts()
    .rename_axis("feature_group")
    .reset_index(name="feature_count")
)

feature_set_summary = pd.DataFrame([
    {"item": "Rows", "value": X_all.shape[0]},
    {"item": "All/combined feature count", "value": X_all.shape[1]},
    {"item": "Champion feature count", "value": sum(c.startswith(("BlueChamp_", "RedChamp_")) for c in all_feature_cols)},
    {"item": "Blue win ratio", "value": y_all.mean()},
    {"item": "Primary metric", "value": "AUC-ROC"},
])

display(feature_set_summary)
display(feature_group_table)


## 6. Train / Validation / Test Split

แบ่งข้อมูลเป็น train, validation และ test แบบ stratified และสร้างทั้งชุด scaled และ unscaled สำหรับโมเดลที่ต้องการ preprocessing ต่างกัน


In [ ]:
def prepare_data(X, y, val_size=0.15, test_size=0.15, random_state=1337, scale=False):
    temp_size = val_size + test_size
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=temp_size, random_state=random_state, stratify=y
    )

    relative_test_size = test_size / temp_size
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=relative_test_size, random_state=random_state, stratify=y_temp
    )

    scaler = None
    X_train_sc, X_val_sc, X_test_sc = X_train, X_val, X_test

    if scale:
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_val_sc   = scaler.transform(X_val)
        X_test_sc  = scaler.transform(X_test)

    return {
        "X_train": X_train_sc,
        "X_val": X_val_sc,
        "X_test": X_test_sc,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "scaler": scaler,
    }

data_sc = prepare_data(X_all, y_all, scale=True)
data = prepare_data(X_all, y_all, scale=False)

X_train_sc, X_val_sc, X_test_sc = data_sc["X_train"], data_sc["X_val"], data_sc["X_test"]
X_train, X_val, X_test = data["X_train"], data["X_val"], data["X_test"]
y_train, y_val, y_test = data["y_train"], data["y_val"], data["y_test"]

X_train.shape, X_val.shape, X_test.shape


## 7. Hyperparameter Tuning

นิยาม objective functions ของแต่ละโมเดลสำหรับ Optuna โดยใช้ validation AUC เป็นค่าที่ optimize และตั้งค่า GPU fallback สำหรับ XGBoost, LightGBM และ CatBoost


In [ ]:
N_TRIALS = 50
USE_GPU = True
GPU_FALLBACK_TO_CPU = True

def gpu_params_for(model_name):
    if not USE_GPU:
        return {}

    if model_name == "XGBoost":
        return {"tree_method": "hist", "device": "cuda"}
    if model_name == "LightGBM":
        return {"device_type": "gpu"}
    if model_name == "CatBoost":
        return {"task_type": "GPU", "devices": "0"}
    return {}

def fit_with_optional_gpu_fallback(model_name, build_model, fit_fn):
    model = build_model(use_gpu=USE_GPU)

    try:
        fit_fn(model)
        return model
    except Exception as exc:
        can_fallback = (
            USE_GPU
            and GPU_FALLBACK_TO_CPU
            and model_name in {"XGBoost", "LightGBM", "CatBoost"}
        )

        if not can_fallback:
            raise

        print(f"GPU failed for {model_name}; retrying on CPU.")
        print(f"Reason: {type(exc).__name__}: {exc}")

        model = build_model(use_gpu=False)
        fit_fn(model)
        return model

def obj_logreg(trial):
    m = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        penalty="l2",
        max_iter=500,
        random_state=42,
    )

    m.fit(X_train_sc, y_train)
    return roc_auc_score(y_val, m.predict_proba(X_val_sc)[:, 1])

def obj_dt(trial):
    m = DecisionTreeClassifier(
        max_depth=trial.suggest_int("max_depth", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 30),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        random_state=42,
    )

    m.fit(X_train.values, y_train.values)
    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])

def obj_rf(trial):
    m = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 300, step=50),
        max_depth=trial.suggest_int("max_depth", 6, 16),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 2, 20),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        bootstrap=True,
        max_samples=trial.suggest_categorical("max_samples", [0.5, 0.75]),
        random_state=42,
        n_jobs=-1,
    )

    m.fit(X_train.values, y_train.values)
    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])

def obj_xgb(trial):
    params = {
        "n_estimators": 800,
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        "eval_metric": "logloss",
        "early_stopping_rounds": 50,
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": 0,
    }

    def build_model(use_gpu):
        if use_gpu:
            gpu_params = {**gpu_params_for("XGBoost"), "sampling_method": "gradient_based"}
        else:
            gpu_params = {"tree_method": "hist", "sampling_method": "uniform"}
        return xgb.XGBClassifier(**params, **gpu_params)

    m = fit_with_optional_gpu_fallback(
        "XGBoost",
        build_model,
        lambda model: model.fit(
            X_train.values,
            y_train.values,
            eval_set=[(X_val.values, y_val.values)],
            verbose=False,
        ),
    )

    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])

def obj_lgb(trial):
    params = {
        "n_estimators": 800,
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    def build_model(use_gpu):
        gpu_params = gpu_params_for("LightGBM") if use_gpu else {}
        return lgb.LGBMClassifier(**params, **gpu_params)

    m = fit_with_optional_gpu_fallback(
        "LightGBM",
        build_model,
        lambda model: model.fit(
            X_train.values,
            y_train.values,
            eval_set=[(X_val.values, y_val.values)],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        ),
    )

    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])

def obj_catboost(trial):
    params = {
        "iterations": 800,
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5, log=True),
        "random_seed": 42,
        "verbose": 0,
        "early_stopping_rounds": 50,
    }

    def build_model(use_gpu):
        gpu_params = gpu_params_for("CatBoost") if use_gpu else {}
        return CatBoostClassifier(**params, **gpu_params)

    m = fit_with_optional_gpu_fallback(
        "CatBoost",
        build_model,
        lambda model: model.fit(
            X_train.values,
            y_train.values,
            eval_set=(X_val.values, y_val.values),
        ),
    )

    return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])


## 8. Run Tuning

รัน Optuna สำหรับแต่ละโมเดล แล้วเก็บ best validation AUC และ best parameters


In [ ]:
studies = {}
objectives = {
    "LogisticRegression": obj_logreg,
    "DecisionTree": obj_dt,
    "RandomForest": obj_rf,
    "XGBoost": obj_xgb,
    "LightGBM": obj_lgb,
    "CatBoost": obj_catboost,
}

MODEL_TRIALS = {
    "LogisticRegression": 1,
    "DecisionTree": 10,
    "RandomForest": 10,
    "XGBoost": 10,
    "LightGBM": 10,
    "CatBoost": 10,
}

for name, obj_fn in objectives.items():
    n_trials = MODEL_TRIALS.get(name, N_TRIALS)

    print(f"\n  > Tuning {name} ({n_trials} trials) ...")
    study = optuna.create_study(direction="maximize", study_name=name)
    study.optimize(obj_fn, n_trials=n_trials, show_progress_bar=False)
    studies[name] = study

    print(f"Best Val AUC: {study.best_value:.4f}")
    print(f"Best Params:  {study.best_params}")


## 9. Retrain Tuned Models And Evaluate On Test Set

สร้างโมเดลใหม่จาก best parameters ของแต่ละ study, train อีกครั้ง และประเมินผลบน test set


In [ ]:
def build_best_model(name, params, use_gpu=USE_GPU):
    if name == "LogisticRegression":
        return LogisticRegression(C=1.0, solver="lbfgs", penalty="l2", max_iter=500, random_state=42)
    if name == "DecisionTree":
        return DecisionTreeClassifier(**params, random_state=42)
    if name == "RandomForest":
        return RandomForestClassifier(**params, bootstrap=True, random_state=42, n_jobs=-1)
    if name == "XGBoost":
        if use_gpu:
            gpu_params = {**gpu_params_for("XGBoost"), "sampling_method": "gradient_based"}
        else:
            gpu_params = {"tree_method": "hist", "sampling_method": "uniform"}
        return xgb.XGBClassifier(
            **params,
            n_estimators=800,
            eval_metric="logloss",
            early_stopping_rounds=50,
            random_state=42,
            n_jobs=-1,
            verbosity=0,
            **gpu_params,
        )
    if name == "LightGBM":
        gpu_params = gpu_params_for("LightGBM") if use_gpu else {}
        return lgb.LGBMClassifier(
            **params,
            n_estimators=800,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
            **gpu_params,
        )
    if name == "CatBoost":
        gpu_params = gpu_params_for("CatBoost") if use_gpu else {}
        return CatBoostClassifier(
            **params,
            iterations=800,
            random_seed=42,
            verbose=0,
            early_stopping_rounds=50,
            **gpu_params,
        )
    raise ValueError(f"Unknown model name: {name}")

results = {}

for name, study in studies.items():
    bp = study.best_params.copy()
    uses_scaled = name == "LogisticRegression"
    Xtr = X_train_sc if uses_scaled else X_train.values
    Xv = X_val_sc if uses_scaled else X_val.values
    Xte = X_test_sc if uses_scaled else X_test.values

    def fit_model(model):
        if name == "XGBoost":
            model.fit(Xtr, y_train.values, eval_set=[(Xv, y_val.values)], verbose=False)
        elif name == "LightGBM":
            model.fit(
                Xtr,
                y_train.values,
                eval_set=[(Xv, y_val.values)],
                callbacks=[lgb.early_stopping(50, verbose=False)],
            )
        elif name == "CatBoost":
            model.fit(Xtr, y_train.values, eval_set=(Xv, y_val.values))
        else:
            model.fit(Xtr, y_train.values)

    model = fit_with_optional_gpu_fallback(
        name,
        lambda use_gpu: build_best_model(name, bp, use_gpu=use_gpu),
        fit_model,
    )

    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    results[name] = {
        "model": model,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "acc": accuracy_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, y_prob),
        "f1": f1_score(y_test, y_pred),
        "best_params": study.best_params,
        "val_auc": study.best_value,
    }

    print(f"\n> {name}")
    print(f"  Val AUC:  {study.best_value:.4f}")
    print(f"  Test Acc: {results[name]['acc']:.4f} | Test AUC: {results[name]['auc']:.4f} | Test F1: {results[name]['f1']:.4f}")


## 10. Numeric Model Comparison

สร้างตาราง metric ของทุกโมเดล, เลือก best model จาก `Test_AUC`, และเปรียบเทียบกับ Logistic Regression


In [ ]:
OUTPUT_DIR = Path("outputs/all_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame({
    name: {
        "Val_AUC": r["val_auc"],
        "Test_Accuracy": r["acc"],
        "Test_AUC": r["auc"],
        "Test_F1": r["f1"],
    }
    for name, r in results.items()
}).T.sort_values("Test_AUC", ascending=False)

summary["AUC_Rank"] = summary["Test_AUC"].rank(ascending=False, method="min").astype(int)
summary["Gap_vs_Best_AUC"] = summary["Test_AUC"].max() - summary["Test_AUC"]
summary["Gap_vs_LogReg_AUC"] = summary["Test_AUC"] - summary.loc["LogisticRegression", "Test_AUC"]

display(summary)
summary.to_csv(OUTPUT_DIR / "summary.csv")
summary.to_csv(OUTPUT_DIR / "model_numeric_comparison.csv")

params_df = pd.DataFrame({name: r["best_params"] for name, r in results.items()}).T
params_df.to_csv(OUTPUT_DIR / "best_params.csv")

best_name = summary.index[0]
best_result = results[best_name]

key_model_names = ["LogisticRegression"]
if best_name not in key_model_names:
    key_model_names.append(best_name)

key_model_summary = summary.loc[key_model_names, [
    "AUC_Rank",
    "Val_AUC",
    "Test_AUC",
    "Test_Accuracy",
    "Test_F1",
    "Gap_vs_Best_AUC",
    "Gap_vs_LogReg_AUC",
]]
display(key_model_summary)
key_model_summary.to_csv(OUTPUT_DIR / "logreg_vs_best_summary.csv")

print(f"Best model by Test AUC: {best_name}")
print(f"Logistic Regression Test AUC: {summary.loc['LogisticRegression', 'Test_AUC']:.4f}")
print(f"Best Model Test AUC: {best_result['auc']:.4f}")
print(f"AUC gap (Best - Logistic): {best_result['auc'] - summary.loc['LogisticRegression', 'Test_AUC']:+.4f}")

AGG_SUMMARY_PATH = Path("outputs/aggregate_only/summary.csv")
aggregate_comparison = None

if AGG_SUMMARY_PATH.exists():
    agg_summary = pd.read_csv(AGG_SUMMARY_PATH, index_col=0)
    agg_best_name = agg_summary["Test_AUC"].idxmax()
    aggregate_comparison = pd.DataFrame([
        {
            "Feature_Set": "Aggregate-Only",
            "Best_Model": agg_best_name,
            "Test_AUC": agg_summary.loc[agg_best_name, "Test_AUC"],
            "Test_Accuracy": agg_summary.loc[agg_best_name, "Test_Accuracy"],
            "Test_F1": agg_summary.loc[agg_best_name, "Test_F1"],
        },
        {
            "Feature_Set": "All / Combined Features",
            "Best_Model": best_name,
            "Test_AUC": best_result["auc"],
            "Test_Accuracy": best_result["acc"],
            "Test_F1": best_result["f1"],
        },
    ])
    aggregate_comparison["Delta_vs_Aggregate_AUC"] = aggregate_comparison["Test_AUC"] - aggregate_comparison.loc[0, "Test_AUC"]
    display(aggregate_comparison)
    aggregate_comparison.to_csv(OUTPUT_DIR / "comparison_to_aggregate.csv", index=False)
else:
    print("Aggregate-only summary not found in this session.")
    print("Run 04_modeling_aggregate_only_kaggle.ipynb first, or compare manually with outputs/aggregate_only/summary.csv after both notebooks finish.")


## 11. Model Comparison Visualization

สร้าง grouped bar chart ของ `Test_AUC`, `Test_Accuracy`, `Test_F1` และ plot ROC/confusion matrix ของ best model


In [ ]:
metrics = ["Test_AUC", "Test_Accuracy", "Test_F1"]
summary_long = (
    summary
    .reset_index()
    .rename(columns={"index": "Model"})
    .melt(
        id_vars="Model",
        value_vars=metrics,
        var_name="Metric",
        value_name="Score",
    )
)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=summary_long, x="Model", y="Score", hue="Metric")
ax.set_title("All-Features Model Comparison")
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.tick_params(axis="x", rotation=30)
ax.set_ylim(0, 1)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

key_plot = key_model_summary.reset_index().rename(columns={"index": "Model"})
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=key_plot, x="Model", y="Test_AUC", color="#F28E2B")
ax.set_title("Logistic Regression vs Best Model")
ax.set_xlabel("")
ax.set_ylabel("Test AUC")
ax.set_ylim(0, 1)
for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", fontsize=10, padding=3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "logreg_vs_best_auc.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_test, best_result["y_prob"], ax=axes[0])
axes[0].set_title(f"ROC Curve: {best_name}")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_result["y_pred"],
    display_labels=["Red Win", "Blue Win"],
    ax=axes[1],
    cmap="Blues",
)
axes[1].set_title(f"Confusion Matrix: {best_name}")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "best_model_eval.png", dpi=150, bbox_inches="tight")
plt.show()

print("Classification Report: Best Model")
print(classification_report(y_test, best_result["y_pred"], target_names=["Red Win", "Blue Win"]))

if best_name != "LogisticRegression":
    print("\nClassification Report: Logistic Regression")
    print(classification_report(y_test, results["LogisticRegression"]["y_pred"], target_names=["Red Win", "Blue Win"]))


## 12. Feature Importance

คำนวณ feature importance หรือ coefficient ของ Logistic Regression และ best model พร้อมสรุป importance ตามกลุ่ม feature


In [ ]:
def build_feature_signal_table(model_name, feature_cols, feature_group_fn):
    model = results[model_name]["model"]
    if hasattr(model, "coef_"):
        table = pd.DataFrame({
            "model": model_name,
            "feature": feature_cols,
            "raw_value": model.coef_[0],
        })
        table["importance"] = table["raw_value"].abs()
        table["direction"] = np.where(table["raw_value"] >= 0, "Blue-favored", "Red-favored")
        table["importance_type"] = "absolute_logistic_coefficient"
    elif hasattr(model, "feature_importances_"):
        table = pd.DataFrame({
            "model": model_name,
            "feature": feature_cols,
            "raw_value": model.feature_importances_,
            "importance": model.feature_importances_,
        })
        table["direction"] = "higher importance"
        table["importance_type"] = "tree_feature_importance"
    else:
        return None

    table["feature_group"] = table["feature"].map(feature_group_fn)
    table = table.sort_values("importance", ascending=False).reset_index(drop=True)
    table["importance_rank"] = np.arange(1, len(table) + 1)
    return table

feature_signal_tables = []
for model_name in key_model_names:
    table = build_feature_signal_table(model_name, all_feature_cols, feature_group)
    if table is not None:
        feature_signal_tables.append(table)

if feature_signal_tables:
    feature_importance_df = pd.concat(feature_signal_tables, ignore_index=True)
    feature_importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

    for model_name in key_model_names:
        model_signals = feature_importance_df[feature_importance_df["model"] == model_name].head(25)
        if model_signals.empty:
            continue

        display(model_signals[[
            "model", "importance_rank", "feature", "importance", "raw_value", "direction", "feature_group"
        ]])

        plt.figure(figsize=(10, 8))
        sns.barplot(data=model_signals, x="importance", y="feature", hue="feature_group", dodge=False)
        plt.title(f"Top Feature Importance: {model_name}")
        plt.xlabel("Importance")
        plt.ylabel("")
        plt.tight_layout()
        safe_name = model_name.lower().replace(" ", "_")
        plt.savefig(OUTPUT_DIR / f"feature_importance_{safe_name}.png", dpi=150, bbox_inches="tight")
        plt.show()

    group_importance = (
        feature_importance_df
        .groupby(["model", "feature_group"], as_index=False)["importance"]
        .sum()
        .sort_values(["model", "importance"], ascending=[True, False])
    )
    group_importance["share"] = group_importance.groupby("model")["importance"].transform(lambda s: s / s.sum())
    group_importance.to_csv(OUTPUT_DIR / "feature_group_importance.csv", index=False)

    for model_name in key_model_names:
        model_group = group_importance[group_importance["model"] == model_name]
        if model_group.empty:
            continue

        display(model_group)

        plt.figure(figsize=(9, 5))
        sns.barplot(data=model_group, x="importance", y="feature_group", color="#59A14F")
        plt.title(f"Feature Group Importance: {model_name}")
        plt.xlabel("Total importance")
        plt.ylabel("")
        plt.tight_layout()
        safe_name = model_name.lower().replace(" ", "_")
        plt.savefig(OUTPUT_DIR / f"feature_group_importance_{safe_name}.png", dpi=150, bbox_inches="tight")
        plt.show()

    generic_model = best_name if best_name in key_model_names else key_model_names[0]
    generic_signals = feature_importance_df[feature_importance_df["model"] == generic_model].head(25)
    if not generic_signals.empty:
        plt.figure(figsize=(10, 8))
        sns.barplot(data=generic_signals, x="importance", y="feature", hue="feature_group", dodge=False)
        plt.title(f"Top Feature Importance: {generic_model}")
        plt.xlabel("Importance")
        plt.ylabel("")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Selected models do not expose coefficients or feature_importances_.")


## 13. Train Final Model

train final model ด้วยข้อมูล train และ validation รวมกัน แล้วประเมินบน test set เดิม พร้อมบันทึก metric และ model artifact


In [ ]:
def best_iteration_from_fitted_model(name, fitted_model):
    if name == "XGBoost":
        best_iteration = getattr(fitted_model, "best_iteration", None)
        return None if best_iteration is None else int(best_iteration) + 1
    if name == "LightGBM":
        best_iteration = getattr(fitted_model, "best_iteration_", None)
        return None if best_iteration is None or best_iteration <= 0 else int(best_iteration)
    if name == "CatBoost":
        best_iteration = fitted_model.get_best_iteration()
        return None if best_iteration is None else int(best_iteration) + 1
    return None

def build_final_model(name, params, tuned_model=None, use_gpu=USE_GPU):
    params = params.copy()
    best_n_estimators = best_iteration_from_fitted_model(name, tuned_model)

    if name == "LogisticRegression":
        return LogisticRegression(C=1.0, solver="lbfgs", penalty="l2", max_iter=500, random_state=42)
    if name == "DecisionTree":
        return DecisionTreeClassifier(**params, random_state=42)
    if name == "RandomForest":
        return RandomForestClassifier(**params, bootstrap=True, random_state=42, n_jobs=-1)
    if name == "XGBoost":
        params["n_estimators"] = best_n_estimators or 800
        if use_gpu:
            gpu_params = {**gpu_params_for("XGBoost"), "sampling_method": "gradient_based"}
        else:
            gpu_params = {"tree_method": "hist", "sampling_method": "uniform"}
        return xgb.XGBClassifier(**params, eval_metric="logloss", random_state=42, n_jobs=-1, verbosity=0, **gpu_params)
    if name == "LightGBM":
        params["n_estimators"] = best_n_estimators or 800
        gpu_params = gpu_params_for("LightGBM") if use_gpu else {}
        return lgb.LGBMClassifier(**params, random_state=42, n_jobs=-1, verbose=-1, **gpu_params)
    if name == "CatBoost":
        params["iterations"] = best_n_estimators or 800
        gpu_params = gpu_params_for("CatBoost") if use_gpu else {}
        return CatBoostClassifier(**params, random_seed=42, verbose=0, **gpu_params)
    raise ValueError(f"Unknown model name: {name}")

uses_scaled = best_name == "LogisticRegression"
if uses_scaled:
    X_train_final = np.vstack([X_train_sc, X_val_sc])
else:
    X_train_final = pd.concat([X_train, X_val], axis=0).values

y_train_final = pd.concat([y_train, y_val], axis=0).values
X_test_final = X_test_sc if uses_scaled else X_test.values

final_model = fit_with_optional_gpu_fallback(
    best_name,
    lambda use_gpu: build_final_model(
        best_name,
        results[best_name]["best_params"],
        tuned_model=results[best_name]["model"],
        use_gpu=use_gpu,
    ),
    lambda model: model.fit(X_train_final, y_train_final),
)

final_pred = final_model.predict(X_test_final)
final_prob = final_model.predict_proba(X_test_final)[:, 1]

final_metrics = pd.DataFrame([{
    "Model": best_name,
    "Feature_Set": "All / Combined Features",
    "Train_Rows": len(y_train_final),
    "Feature_Count": len(all_feature_cols),
    "Test_Accuracy": accuracy_score(y_test, final_pred),
    "Test_AUC": roc_auc_score(y_test, final_prob),
    "Test_F1": f1_score(y_test, final_pred),
}])

display(final_metrics)
final_metrics.to_csv(OUTPUT_DIR / "final_metrics.csv", index=False)
joblib.dump(final_model, OUTPUT_DIR / "final_best_model.joblib")

numeric_conclusion = (
    f"LogisticRegression Test AUC={summary.loc['LogisticRegression', 'Test_AUC']:.4f}; "
    f"Best Model={best_name}; "
    f"Best Model final Test AUC={final_metrics.loc[0, 'Test_AUC']:.4f}; "
    f"AUC gap={final_metrics.loc[0, 'Test_AUC'] - summary.loc['LogisticRegression', 'Test_AUC']:+.4f}; "
    f"Best Model Accuracy={final_metrics.loc[0, 'Test_Accuracy']:.4f}; "
    f"Best Model F1={final_metrics.loc[0, 'Test_F1']:.4f}."
)

if "aggregate_comparison" in globals() and aggregate_comparison is not None:
    numeric_conclusion += f" All-features vs aggregate-only AUC delta={aggregate_comparison.loc[1, 'Delta_vs_Aggregate_AUC']:+.4f}."

print("\nNumeric Conclusion:")
print(numeric_conclusion)
(OUTPUT_DIR / "conclusion.txt").write_text(numeric_conclusion, encoding="utf-8")

print(f"Final model saved to: {OUTPUT_DIR / 'final_best_model.joblib'}")
print(f"Final metrics saved to: {OUTPUT_DIR / 'final_metrics.csv'}")


## 14. Output Files

cell ก่อนหน้าจะบันทึกผลลัพธ์ลง `outputs/all_features` เช่น metric tables, model comparison plot, ROC/confusion matrix, feature importance, feature group importance, final metrics และ serialized model
